# Park 剩餘26段｜已核准九句直接內附並鎖定
目標10/30 Windows三關整合。這份新輸入包含九句全部accepted原音，不需重上傳九句、不需重選角。免費T4，1→2→3核對dry-run，再手動開4生成26，第5逐句驗收。

## 1｜安裝（免費T4 GPU，沿已成功流程）

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2｜只上傳 park-remaining26-inputs.zip

In [ ]:
from google.colab import files,output as colab_output
from pathlib import Path
import zipfile,io,hashlib,importlib.util,sys,uuid
uploaded=files.upload()
if len(uploaded)!=1:raise ValueError('只上傳park-remaining26-inputs.zip；不需要另外上傳九句結果')
raw=next(iter(uploaded.values()))
EXPECTED_TOOLS={'park_tools.py': '30e82f59aaec5edccb78b055d42bcf65e6d36df26702477e99226d503388cb43', 'remaining26_runner.py': 'e599ecab663e8477b67cbe6bd9e3ec91ed113fdb9ca0239f469702b2d0aeb691'}
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    if len(z.infolist())>300 or sum(i.file_size for i in z.infolist())>100000000:raise ValueError('ZIP異常')
    sources={}
    for name,h in EXPECTED_TOOLS.items():
        if z.namelist().count(name)!=1:raise ValueError('工具檔數量不符')
        data=z.read(name)
        if hashlib.sha256(data).hexdigest()!=h:raise ValueError('工具SHA256不符，請使用本次輸入ZIP')
        sources[name]=data
spec=importlib.util.spec_from_loader('park_tools',loader=None)
park=importlib.util.module_from_spec(spec);sys.modules['park_tools']=park
exec(compile(sources['park_tools.py'],'park_tools.py','exec'),park.__dict__)
spec=importlib.util.spec_from_loader('park_remaining26_locked',loader=None)
runner=importlib.util.module_from_spec(spec)
exec(compile(sources['remaining26_runner.py'],'remaining26_runner.py','exec'),runner.__dict__)
WORK=park.unpack(raw,Path('/content')/('park26_inputs_'+uuid.uuid4().hex[:8]))
runner.plan(WORK)
OUTPUT=Path('/content')/('park_complete36_'+uuid.uuid4().hex[:8])
colab_output.enable_custom_widget_manager()
from IPython.display import display,Markdown
print('核准九句、Library共用音、參考音與逐字稿全部核對通過；未生成。')

## 3｜26句dry-run，九句SHA鎖定

In [ ]:
dry=runner.generate(WORK,OUTPUT,run=False)
rows=['|角色|聲線|lineId|英文原文|','|---|---|---|---|']
for a in dry['jobs']:rows.append('|'+a['role']+'|'+a['voiceId']+'|'+a['lineId']+'|'+a['text']+'|')
display(Markdown('\n'.join(rows)))
print('只生成26句：Rosie10、Pirate16；九句+Library共用1保留原WAV。generated=0。')

## 4｜只生成26句（不重生九句或共用音）
核對清單後設定RUN_GENERATION=True與CONFIRMATION=CONFIRM_PARK_REMAINING_26。相同OUTPUT重跑只補缺句。

In [ ]:
RUN_GENERATION=False
CONFIRMATION=''
if RUN_GENERATION:
    print(runner.generate(WORK,OUTPUT,run=True,confirmation=CONFIRMATION))
else:print('尚未生成，開關預設關閉。')

## 5｜完整36段逐句驗收與ZIP下載
九句已接受與Library共用1鎖定，只有26新句可選接受／重錄／未決定及備註。自動保存，JSON可匯出/載入。需要重錄只建立ID清單，不立即推論。生成若中斷，這格也可保存已完成部分。

In [ ]:
if (OUTPUT/'generated-manifest.json').exists():
    PARK_REVIEW_UI=runner.show_review(OUTPUT)
else:print('尚無生成結果。可用第6格恢復已有結果ZIP；本格不生成。')

## 6｜可選：結果ZIP恢復／續跑
上傳26段批次的部分或完整結果ZIP，恢復OUTPUT與驗收選擇；生成續跑需要第2格WORK。若只試聽，用獨立CPU恢復Notebook，不需TTS安裝。

In [ ]:
resume_upload=files.upload()
if len(resume_upload)!=1:raise ValueError('只上傳一個Park remaining26結果ZIP')
OUTPUT=runner.restore(next(iter(resume_upload.values())),Path('/content'))
PARK_REVIEW_UI=runner.show_review(OUTPUT)
print('已恢復；若要補缺句，再執行第4格，只生成尚缺音檔。')

## 範圍與費用
新26只有Rosie10/Pirate16。Mermaid af_nicole與ring am_onyx已在核准九句內，固定保留，不需新生成。全Park36=已接受九句+新26+Library共用1。核准參考、逐字稿、模型revision不變，無pitchDSP、無付費API/訂閱。免費T4有配額/等待限制，模型下載數GB。原83核准音與穩定版/正式遊戲不改。

Colab暫存可能清除，離開前下載ZIP。已做輸入/鎖定/續跑/驗收測試；本機未推論這26句，不宣稱音質已接受。